## 2.2. Task Submission - City Bike New York

In [ ]:
!pip install pyarrow

### importing libraries:

In [ ]:
#importing libraries
import pandas as pd
import numpy as np
import os
import requests
import json
from datetime import datetime

print("pandas:", pd.__version__)
print("requests:", requests.__version__)
print("json: OK")

##### Some notes:
- my computer crashed with these files. it doesn't have enough RAM. So I proceeded with Kaggle, which offers me 30gb instead of my original 8gb.
- Some constrictions with CPU were found, but manageable: Had to discard three columns, ride_id, start_station_id e end_station_id.

### uploading and checking citibike files:

In [ ]:
# checking columns available in one CSV file
sample_path = '/kaggle/input/datasets/danielabranca/full-data-set/2022-citibike-tripdata/202201-citibike-tripdata/202201-citibike-tripdata_1.csv'
df_sample = pd.read_csv(sample_path, nrows=1)
print(df_sample.columns.tolist())

In [ ]:
df_sample.head()

In [ ]:
#this is a file explorer: it runs all folders and sub folders and prints the complete path for each file.
for root, dirs, files in os.walk('/kaggle/input/datasets/danielabranca/full-data-set'):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
#By uploading a zip folder to Kaggle, Kaggle will unzip them in the process. Therefore, the following code applies:

#defining the path/folder with all csvs:
folderpath = '/kaggle/input/datasets/danielabranca/full-data-set/2022-citibike-tripdata'

#defining the columns that will be uploaded (due to RAM constrictions):
cols = ['rideable_type', 'started_at', 'ended_at', 'start_station_name', 
        'end_station_name', 'start_lat', 'start_lng', 
        'end_lat', 'end_lng', 'member_casual']

#creating an empty dataframe that will grow as files are added:
df = pd.DataFrame()

#it runs over all folders and sub folders and filters only csv files:
for root, dirs, files in os.walk(folderpath):
    for file in files:
        if file.endswith('.csv'):
            filepath = os.path.join(root, file) #it builds the complete path for each file
            df_temp = pd.read_csv(filepath, usecols=cols) #it reads the csv reading only the selected columns
            df = pd.concat([df, df_temp], ignore_index=True) #it adds the new file to the main dataframe
            del df_temp #it deletes the temporary dataframe to release memory immediately (useful when there's CPU constrictions)
            print(f"Read: {file}") #it confirms the file was read.

print(f"\nTotal of lines: {len(df)}")

In [ ]:
#understanding the dataframe:
df.shape

In [ ]:
df.head()

In [ ]:
df.tail()

### Getting weather data using NOAA's API

##### Some notes: 
- I downloaded the weather dataset from NOAA's website because it was easier, since in Kaggle it isn't possible to use API. I'll upload it here as a csv.
- There was no TAVG, so I calculated it and created a new column for that purpose.

### If API would work, this is how I'd do it:

````python
#defining my NOAA token:
Token = 'PRqSNMTFOrNOzcjYCmZrbQQGnZnKMGoc'

````python
r = requests.get('https://www.ncei.noaa.gov/cdo-web/api/v2/data?datasetid=GHCND&datatypeid=TMAX&datatypeid=TMIN&datatypeid=TAVG&limit=1000&stationid=GHCND:USW00014732&startdate=2022-01-01&enddate=2022-12-31&units=metric', headers={'token': 'PRqSNMTFOrNOzcjYCmZrbQQGnZnKMGoc'})

d = json.loads(r.text)

````python
d = r.json()

#split TMAX e TMIN
tmax = [item for item in d['results'] if item['datatype'] == 'TMAX']
tmin = [item for item in d['results'] if item['datatype'] == 'TMIN']

#extract dates and values
dates = [item['date'] for item in tmax]
temps_max = [item['value'] for item in tmax]
temps_min = [item['value'] for item in tmin]

#calculate average
temps_avg = [(mx + mn) / 2 for mx, mn in zip(temps_max, temps_min)]

```python
print(len(dates), len(temps_avg))

````python
from datetime import datetime
#putting the results in a dataframe, converting it to proper formats; 
df_temp = pd.DataFrame()
df_temp['date'] = [datetime.strptime(d, "%Y-%m-%dT%H:%M:%S") for d in dates]
df_temp['avgTemp'] = temps_avg

### Uploading CSV from NOAA:
 - Kaggle doesn't allow access to other websites by default. So let's upload this in another way, by downloading the weather dataset directly from NOAA and uploading it here:

In [ ]:
#lets upload the csv weather file to the notebook:
df_temp = pd.read_csv(r"/kaggle/input/datasets/danielabranca/data-weather/weather_ny_2022.csv")

In [ ]:
df_temp.head()

In [ ]:
#calculating TAVG and standardizing all variables with ºC and round 0:
df_temp['TMAX'] = ((df_temp['TMAX'] - 32) * 5/9).round(0).astype(int)
df_temp['TMIN'] = ((df_temp['TMIN'] - 32) * 5/9).round(0).astype(int)
df_temp['TAVG'] = (((df_temp['TMAX'] + df_temp['TMIN']) / 2 - 32) * 5/9).round(0).astype(int)
df_temp

In [ ]:
#checking string types:
df_temp.dtypes

In [ ]:
df.dtypes

In [ ]:
#converting started_at and ended_at to datetime:
df['started_at'] = pd.to_datetime(df['started_at'], format='mixed')
df['ended_at'] = pd.to_datetime(df['ended_at'], format='mixed')

In [ ]:
#deriving start_date column with date only:
df['start_date'] = df['started_at'].dt.date
df['start_date'] = pd.to_datetime(df['start_date'])

In [ ]:
df.dtypes

In [ ]:
#converting date from df into datetime as well:
df_temp['DATE'] = pd.to_datetime(df_temp['DATE'])

In [ ]:
df_temp.dtypes

In [ ]:
#doing the merge:
df_merged = df.merge(df_temp, how='left', left_on='start_date', right_on='DATE', indicator =True)
print(df_merged['_merge'].value_counts())

In [ ]:
#seeing lines with no correspondence (even though it's neglectible):
df_missing = df_merged[df_merged['_merge'] == 'left_only']
print(df_missing['start_date'].value_counts())

In [ ]:
#saving this as pq file:
df_merged.to_parquet('/kaggle/working/citibike_merged_2022.parquet', index=False)

### Pulling files to git hub repo:

In [1]:
#adding to github repo
!git add 2.2.city_bike_final.ipynb
!git commit -m "Add notebook on API data access and merging"
!git push -u origin main

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
